# 00 — CNN image classification with PyTorch

PyTorch-only track. Dataset: CIFAR-10, automatically downloaded by torchvision. For Homer/Bart, use the custom-folder dataset contract below.

## 1. Environment and version

In [ ]:
import torch, torchvision
print('PyTorch:',torch.__version__,'torchvision:',torchvision.__version__)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)
assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) >= (2,2)

## 2. Data understanding and preprocessing

For an image tensor, shape is `(C,H,W)`; a batch is `(N,C,H,W)`. Normalize each channel with \(x'=(x-\\mu)/\\sigma\).

In [ ]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,random_split
import torch.nn as nn, torch.optim as optim
import numpy as np
MEAN=(.4914,.4822,.4465); STD=(.247,.243,.261)
train_tf=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
raw=datasets.CIFAR10('data',train=True,download=True)
print('Classes:',raw.classes,'count:',len(raw),'sample size:',raw[0][0].size)
train_base=datasets.CIFAR10('data',train=True,download=True,transform=train_tf)
val_base=datasets.CIFAR10('data',train=True,download=True,transform=eval_tf)
train_idx,val_idx=random_split(range(len(raw)),[45000,5000],generator=torch.Generator().manual_seed(42))
# Use Subset objects backed by independent dataset transforms.
from torch.utils.data import Subset
train_ds=Subset(train_base,train_idx.indices); val_ds=Subset(val_base,val_idx.indices)
test_ds=datasets.CIFAR10('data',train=False,download=True,transform=eval_tf)
train_loader=DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=2); val_loader=DataLoader(val_ds,batch_size=256); test_loader=DataLoader(test_ds,batch_size=256)

## 3. CNN architecture

In [ ]:
class SmallCNN(nn.Module):
 def __init__(self,n_classes=10):
  super().__init__(); self.features=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(128,n_classes)
 def forward(self,x): return self.head(self.features(x).flatten(1))
model=SmallCNN().to(device); criterion=nn.CrossEntropyLoss(); optimizer=optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
print(model)

## 4. Train and validate

In [ ]:
from sklearn.metrics import accuracy_score
import copy
@torch.no_grad()
def evaluate(loader):
 model.eval(); ys=[]; ps=[]; loss_sum=0
 for x,y in loader:
  x,y=x.to(device),y.to(device); logits=model(x); loss_sum+=criterion(logits,y).item()*len(y); ys.extend(y.cpu().tolist()); ps.extend(logits.argmax(1).cpu().tolist())
 return loss_sum/len(loader.dataset),accuracy_score(ys,ps),ys,ps
best=float('inf'); best_state=None
for epoch in range(10):
 model.train(); total=0
 for x,y in train_loader:
  x,y=x.to(device),y.to(device); optimizer.zero_grad(set_to_none=True); loss=criterion(model(x),y); loss.backward(); optimizer.step(); total+=loss.item()*len(y)
 vl,va,_,_=evaluate(val_loader); print(epoch+1,total/len(train_ds),vl,va)
 if vl<best: best=vl; best_state=copy.deepcopy(model.state_dict())
model.load_state_dict(best_state); torch.save({'model':model.state_dict(),'classes':raw.classes},'cifar10_cnn.pt')

## 5. Final test metrics

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay
import matplotlib.pyplot as plt
_,acc,y_true,y_pred=evaluate(test_loader)
print('Test accuracy:',acc); print(classification_report(y_true,y_pred,target_names=raw.classes))
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,display_labels=raw.classes,xticks_rotation=45); plt.show()

## Tips

- Do not use test metrics for model selection.
- Check image range and tensor layout before training.
- Calculate channel statistics on training data only if dataset-specific normalization is required.
- Verify flips/crops preserve labels for the domain.